In [1]:
!pip install memory-profiler

In [21]:
from typing import Dict, List, Annotated
import numpy as np
import os
import shutil
from sklearn.cluster import MiniBatchKMeans
import pickle
import heapq
import numpy as np

DB_SEED_NUMBER = 42
ELEMENT_SIZE = np.dtype(np.float32).itemsize
DIMENSION = 70


n_clusters_1 = 500
batch_size_1 = 10
nprobe_1 = 30

n_clusters_2 = 100
batch_size_2 = 10
nprobe_2 = 50



class VecDB:
    def __init__(self, database_file_path = "saved_db.dat", index_file_path = "index.dat", new_db = True, db_size = None) -> None:
        self.db_path = database_file_path
        self.index_path = index_file_path
        if new_db:
            if db_size is None:
                raise ValueError("You need to provide the size of the database")
            # delete the old DB file if exists
            if os.path.exists(self.db_path):
                os.remove(self.db_path)
            self.vectors = self.generate_database(db_size)

    def generate_database(self, size: int) -> None:
        rng = np.random.default_rng(DB_SEED_NUMBER)
        vectors = rng.random((size, DIMENSION), dtype=np.float32)
        self._write_vectors_to_file(vectors)
        self._build_index()
        return vectors

    def _write_vectors_to_file(self, vectors: np.ndarray) -> None:
        mmap_vectors = np.memmap(self.db_path, dtype=np.float32, mode='w+', shape=vectors.shape)
        mmap_vectors[:] = vectors[:]
        mmap_vectors.flush()

    def _get_num_records(self) -> int:
        return os.path.getsize(self.db_path) // (DIMENSION * ELEMENT_SIZE)

    def insert_records(self, rows: Annotated[np.ndarray, (int, 70)]):
        num_old_records = self._get_num_records()
        num_new_records = len(rows)
        full_shape = (num_old_records + num_new_records, DIMENSION)
        mmap_vectors = np.memmap(self.db_path, dtype=np.float32, mode='r+', shape=full_shape)
        mmap_vectors[num_old_records:] = rows
        mmap_vectors.flush()
        #TODO: might change to call insert in the index, if you need
        self._build_index()

    def get_one_row(self, row_num: int) -> np.ndarray:
        # This function is only load one row in memory
        try:
            offset = row_num * DIMENSION * ELEMENT_SIZE
            mmap_vector = np.memmap(self.db_path, dtype=np.float32, mode='r', shape=(1, DIMENSION), offset=offset)
            return np.array(mmap_vector[0])
        except Exception as e:
            return f"An error occurred: {e}"

    def get_n_rows(self, row_num: int, n: int) -> np.ndarray:
        # This function loads a specified number of rows starting from row_num
        try:
            offset = row_num * DIMENSION * ELEMENT_SIZE
            mmap_vector = np.memmap(self.db_path, dtype=np.float32, mode='r', shape=(n, DIMENSION), offset=offset)
            return np.array(mmap_vector)
        except Exception as e:
            return f"An error occurred: {e}"


    def get_n_random_rows(self, indices) -> np.ndarray:
        try:
            min_idx = indices.min()
            max_idx = indices.max()
            offset = min_idx * DIMENSION * ELEMENT_SIZE
            n_rows = max_idx - min_idx + 1

            mmap_vector = np.memmap(
                self.db_path,
                dtype=np.float32,
                mode='r',
                shape=(n_rows, DIMENSION),
                offset=offset
            )
            relative_indices = indices - min_idx
            return np.array(mmap_vector[relative_indices])
        except Exception as e:
            return f"An error occurred: {e}"


    def get_all_rows(self) -> np.ndarray:
        # Take care this load all the data in memory
        num_records = self._get_num_records()
        vectors = np.memmap(self.db_path, dtype=np.float32, mode='r', shape=(num_records, DIMENSION))
        return np.array(vectors)

    # {
    #     "1st level cluster centroid":{
    #         "2nd level cluster centroid": [ids],
    #         "2nd level cluster centroid":[ids]
    #     },
    #     "1st level cluster centroid":{
    #         "2nd level cluster centroid": [ids],
    #         "2nd level cluster centroid":[ids]
    #     },
    #  }

    def retrieve(self, query: Annotated[np.ndarray, (1, DIMENSION)], top_k=5):

        with open(self.index_path, 'rb') as index_file:
            cluster_mapping = pickle.load(index_file)

        nearest_neighbors_1 = []
        for centroid_1 in cluster_mapping.keys():
            distance = self._cal_score(query,centroid_1)
            nearest_neighbors_1.append((centroid_1,distance))

        nearest_neighbors_1 = sorted(nearest_neighbors_1, key=lambda x: -x[1])[:nprobe_1]

        nearest_neighbors_2 = []
        for centroid_1,_ in nearest_neighbors_1:
            dict = cluster_mapping[centroid_1]
            for centroid_2 in dict.keys():
                distance = self._cal_score(query,centroid_2)
                nearest_neighbors_2.append((centroid_2,distance))

        nearest_neighbors_2 = sorted(nearest_neighbors_2, key=lambda x: -x[1])[:nprobe_2]


        selected_ids = []
        for centroid_2, _ in nearest_neighbors_2:
            for centroid_1, second_level_dict in cluster_mapping.items():

                if centroid_2 in second_level_dict:
                    selected_ids.extend(second_level_dict[centroid_2])

        indices = np.array(selected_ids)
        rows = self.get_n_random_rows(indices)


        id_distance_pairs = [
            (selected_ids[idx], self._cal_score(query, row))
            for idx, row in enumerate(rows)
        ]

        top_k_results = sorted(id_distance_pairs, key=lambda x: -x[1])[:top_k]

        return [result[0] for result in top_k_results]


    def _cal_score(self, vec1, vec2):
        dot_product = np.dot(vec1, vec2)
        norm_vec1 = np.linalg.norm(vec1)
        norm_vec2 = np.linalg.norm(vec2)
        cosine_similarity = dot_product / (norm_vec1 * norm_vec2)
        return cosine_similarity

    def _build_index(self):
        kmeans = MiniBatchKMeans(n_clusters=n_clusters_1, batch_size=batch_size_1, max_iter=200)

        vectors = self.get_all_rows()

        kmeans.fit(vectors)

        labels = kmeans.predict(vectors)
        centroids = kmeans.cluster_centers_

        cluster_mapping = {tuple(centroid): [] for centroid in centroids}

        for vector_id, label in enumerate(labels):
            centroid_key = tuple(centroids[label])
            cluster_mapping[centroid_key].append(vector_id)

        min_length=float('inf')
        for centroid,vector_ids in cluster_mapping.items():
            if len(vector_ids) < min_length:
                min_length = len(vector_ids)

        n_clusters_2 = min_length


        kmeans_2nd_level = MiniBatchKMeans(n_clusters=n_clusters_2, batch_size=batch_size_2, max_iter=200)

        cluster_mapping_1 = cluster_mapping


        for centroid_1 in cluster_mapping_1.keys():

            cluster_vector_ids = cluster_mapping_1[centroid_1]
            cluster_vector_ids_np = np.array(cluster_vector_ids)

            cluster_vectors = self.get_n_random_rows(cluster_vector_ids_np)

            labels = kmeans_2nd_level.fit_predict(cluster_vectors)
            centroids_2 = kmeans_2nd_level.cluster_centers_


            cluster_mapping_2 = {tuple(centroid_2): [] for centroid_2 in centroids_2}

            for vector_id, label in zip(cluster_vector_ids,labels):
                centroid_2_key = tuple(centroids_2[label])
                cluster_mapping_2[centroid_2_key].append(vector_id)

            cluster_mapping_1[centroid_1] = cluster_mapping_2


        # Save the cluster mapping to a file
        with open(self.index_path, 'wb') as index_file:
            pickle.dump(cluster_mapping_1, index_file)

    # {
    #     "1st level cluster centroid":{
    #         "2nd level cluster centroid": [ids],
    #         "2nd level cluster centroid":[ids]
    #     },
    #     "1st level cluster centroid":{
    #         "2nd level cluster centroid": [ids],
    #         "2nd level cluster centroid":[ids]
    #     },
    #  }

In [22]:
# This snippet of code is to show you a simple evaluate for VecDB class, but the full evaluation for project on the Notebook shared with you.
import numpy as np
import time
from dataclasses import dataclass
from typing import List

@dataclass
class Result:
    run_time: float
    top_k: int
    db_ids: List[int]
    actual_ids: List[int]

def run_queries(db, queries, top_k, actual_ids, num_runs):
    """
    Run queries on the database and record results for each query.

    Parameters:
    - db: Database instance to run queries on.
    - queries: List of query vectors.
    - top_k: Number of top results to retrieve.
    - actual_ids: List of actual results to evaluate accuracy.
    - num_runs: Number of query executions to perform for testing.

    Returns:
    - List of Result
    """
    global results
    results = []
    for i in range(num_runs):
        tic = time.time()
        db_ids = db.retrieve(queries[i], top_k)
        toc = time.time()
        run_time = toc - tic
        results.append(Result(run_time, top_k, db_ids, actual_ids[i]))
    return results


def evaluate_result(results: List[Result]):
    """
    Evaluate the results based on accuracy and runtime.
    Scores are negative. So getting 0 is the best score.

    Parameters:
    - results: A list of Result objects

    Returns:
    - avg_score: The average score across all queries.
    - avg_runtime: The average runtime for all queries.
    """
    scores = []
    run_time = []
    for res in results:
        run_time.append(res.run_time)
        # case for retireving number not equal to top_k, socre will be the lowest
        if len(set(res.db_ids)) != res.top_k or len(res.db_ids) != res.top_k:
            scores.append( -1 * len(res.actual_ids) * res.top_k)
            continue
        score = 0
        for id in res.db_ids:
            try:
                ind = res.actual_ids.index(id)
                if ind > res.top_k * 3:
                    score -= ind
            except:
                score -= len(res.actual_ids)
        scores.append(score)

    return sum(scores) / len(scores), sum(run_time) / len(run_time)

# if __name__ == "__main__":
#     db = VecDB(db_size = 10**6)

#     all_db = db.get_all_rows()

#     res = run_queries(db, all_db, 5, 10)

#     print(eval(res))

In [23]:
from memory_profiler import memory_usage
import gc
import shutil

def memory_usage_run_queries(args):
    """
    Run queries and measure memory usage during the execution.

    Parameters:
    - args: Arguments to be passed to the run_queries function.

    Returns:
    - results: The results of the run_queries.
    - memory_diff: The difference in memory usage before and after running the queries.
    """
    global results
    mem_before = max(memory_usage())
    mem = memory_usage(proc=(run_queries, args, {}), interval = 1e-3)
    return results, max(mem) - mem_before


def get_actual_ids_first_k(actual_sorted_ids, k):
    """
    Retrieve the IDs from the sorted list of actual IDs.
    actual IDs has the top_k for the 20 M database but for other databases we have to remove the numbers higher than the max size of the DB.

    Parameters:
    - actual_sorted_ids: A list of lists containing the sorted actual IDs for each query.
    - k: The DB size.

    Returns:
    - List of lists containing the actual IDs for each query for this DB.
    """
    return [[id for id in actual_sorted_ids_one_q if id < k] for actual_sorted_ids_one_q in actual_sorted_ids]

db_size=10**6
db = VecDB(db_size = db_size)


In [24]:
needed_top_k = 10000
# needed_top_k = 5
QUERY_SEED_NUMBER = 10
rng = np.random.default_rng(QUERY_SEED_NUMBER)
query1 = rng.random((1, 70), dtype=np.float32)
query2 = rng.random((1, 70), dtype=np.float32)
query3 = rng.random((1, 70), dtype=np.float32)
queries = [query1, query2, query3]


actual_sorted_ids_1m_q1 = np.argsort(db.vectors.dot(query1.T).T / (np.linalg.norm(db.vectors, axis=1) * np.linalg.norm(query1)), axis= 1).squeeze().tolist()[::-1][:needed_top_k]
gc.collect()
actual_sorted_ids_1m_q2 = np.argsort(db.vectors.dot(query2.T).T / (np.linalg.norm(db.vectors, axis=1) * np.linalg.norm(query2)), axis= 1).squeeze().tolist()[::-1][:needed_top_k]
gc.collect()
actual_sorted_ids_1m_q3 = np.argsort(db.vectors.dot(query3.T).T / (np.linalg.norm(db.vectors, axis=1) * np.linalg.norm(query3)), axis= 1).squeeze().tolist()[::-1][:needed_top_k]
gc.collect()

actual_sorted_ids_20m = [actual_sorted_ids_1m_q1, actual_sorted_ids_1m_q2, actual_sorted_ids_1m_q3]

query_dummy = rng.random((1, 70), dtype=np.float32)

actual_ids = get_actual_ids_first_k(actual_sorted_ids_20m, db_size)
# print(actual_ids)
res, mem = memory_usage_run_queries((db, queries, 5, actual_ids, 3))
print(res)
# print(res)
eval = evaluate_result(res)


to_print = f"\tscore\t{eval[0]}\ttime\t{eval[1]:.2f}\tRAM\t{mem:.2f} MB"
print(to_print)

[Result(run_time=31.37224531173706, top_k=5, db_ids=[69311, 259570, 82970, 494938, 978033], actual_ids=[69311, 259570, 82970, 494938, 978033, 218341, 255895, 686624, 104914, 347348, 314153, 853839, 83922, 12949, 555107, 612890, 995165, 958462, 275635, 980150, 353395, 743020, 215814, 578966, 799837, 693520, 249091, 793097, 508994, 708701, 535753, 684885, 421787, 874384, 111341, 424783, 23081, 67749, 507348, 833031, 488979, 23239, 924896, 623655, 294364, 513104, 866365, 46224, 917389, 923430, 835153, 551585, 679937, 970377, 631557, 800002, 458918, 716907, 443523, 884921, 515292, 301310, 281440, 67788, 351256, 168566, 952119, 732952, 878445, 24915, 143282, 952732, 864566, 996079, 762441, 928334, 603987, 113899, 229871, 337794, 390703, 806759, 627013, 122260, 180631, 39199, 221611, 959192, 948665, 848251, 133745, 403387, 330505, 716176, 368770, 661961, 968957, 535276, 215540, 42243, 82325, 139613, 316616, 93314, 969287, 926507, 539410, 829944, 622421, 778875, 199223, 986583, 305797, 304288

In [ ]:
shutil.make_archive('clusters1', 'zip', 'clusters1')
shutil.make_archive('clusters2', 'zip', 'clusters2')